# Session 19 / r4 Campaign — FULL launch — v4 (self-contained)

**Thread B (Array-RQMC x collisional PIC), Hyman + Claude, registered 2026-08-12.**

Upload **only this notebook** — both substrates
(`discrepancy_campaign_v1_2_0.py`, sha `a83fc7937442b511`;
`pic1d1v_gate_v1_4_0.py`, sha `dfa9da580ba210ca`) are embedded in
Cell 1, written to the session on first run, and sha-verified against
the registered pins before anything executes. If Colab recycles the
runtime, just Runtime -> Run all again: Cell 1 rewrites the substrates
and the campaign resumes per rung, skipping completed runs.

Runs the r4 registration (session_log entry of record, 2026-08-12):
monotone exact-law coupling arms M4a/M4b (offsets 62-67 / 68-73,
Np ladder 128..8192) and snapshot arms S-C4b/S-C3 (offsets 74-76 /
77-79, Np in {2048, 8192}, npz snapshots at t in {0, 10, 20, 40}).
r3 arms are locked out in code; smoke offsets 0-1 are never used here.
Estimated 1.5-2.5 h on a standard CPU runtime. The final cell packages
96 FULL CSVs + 48 snapshots + MANIFEST (144 files) and downloads the zip.


In [1]:
# Cell 1 - self-contained substrates (v4): write + sha-verify, then env
!pip -q install numpy scipy hilbertcurve
import base64, hashlib
PINS = {"discrepancy_campaign_v1_2_0.py": "a83fc7937442b511", "pic1d1v_gate_v1_4_0.py": "dfa9da580ba210ca"}
B64 = {}
B64["discrepancy_campaign_v1_2_0.py"] = (
"..."
"c3BsaXQoIiwiKSkpCg=="
)
B64["pic1d1v_gate_v1_4_0.py"] = (
"..."
)
for fn, pin in PINS.items():
    data = base64.b64decode(B64[fn])
    open(fn, 'wb').write(data)
    h = hashlib.sha256(data).hexdigest()[:16]
    assert h == pin, f"{fn}: embedded sha {h} != pinned {pin}"
    print('wrote + sha OK ', fn, h)


wrote + sha OK  discrepancy_campaign_v1_2_0.py a83fc7937442b511
wrote + sha OK  pic1d1v_gate_v1_4_0.py dfa9da580ba210ca


In [2]:
# Cell 2 - verification gate: all 10 V-tests must PASS before any campaign run
import discrepancy_campaign_v1_2_0 as H
H.verify()


HARNESS v1.2.0 VERIFICATION SUITE (r4; smoke offsets only)
V0 PASS  gate import regression: rel 1.8% (record: 1.8%)
V1 PASS  Kac invariants over 200 steps: max|P|=2.6e-15, max|dKE|=3.1e-15 (<= 1e-12)
V2 PASS  commensurate-lattice zero: max D = 1.6e-15
V3 PASS  A1-J joint iid plateau: rel 0.06% (< 2%; R=2560)
V4 PASS  h->inf limit reproduces D_K2: |diff| = 5.5e-08
V5 PASS  free-streaming envelope regression: max dev = 3.9e-04
V6 PASS  all ten ECP arms execute (Kac + monotone + projections)
V7 PASS  C2-MCP companion: Dx 1.6e-05 -> 3.3e-05 (growing)
V8 PASS  vectorized Hilbert bit-exact vs reference (4096 pts)
V9 PASS  monotone map exact-law: KE dev 6.7e-16 (< 1e-12); KS 7.50e-04 (< 2.44e-03); moment z 1.49 (< 6); monotone grid OK

All 10 verification tests PASSED.  (135.2s)


In [3]:
# Cell 3 - FULL r4 campaign: M4a, M4b, S-C4b, S-C3 x registered offsets x ladders (resumable)
H.run_registered_campaign("results", full=True)


M4a o62 ladder done (311s)
M4a o63 ladder done (306s)
M4a o64 ladder done (306s)
M4a o65 ladder done (308s)
M4a o66 ladder done (307s)
M4a o67 ladder done (312s)
M4b o68 ladder done (308s)
M4b o69 ladder done (308s)
M4b o70 ladder done (305s)
M4b o71 ladder done (303s)
M4b o72 ladder done (304s)
M4b o73 ladder done (311s)
S-C4b o74 ladder done (225s)
S-C4b o75 ladder done (225s)
S-C4b o76 ladder done (224s)
S-C3 o77 ladder done (227s)
S-C3 o78 ladder done (237s)
S-C3 o79 ladder done (228s)


In [4]:
# Cell 4 - package: SHA-256 manifest + zip + download (expects 96 CSVs + 48 npz)
import hashlib, zipfile, glob, os, datetime
files = sorted(glob.glob("results/*.csv")) + sorted(glob.glob("results/*.npz"))
n_csv = sum(f.endswith(".csv") for f in files)
n_npz = len(files) - n_csv
print(f"inventory: {n_csv} CSVs + {n_npz} snapshots = {len(files)}")
assert (n_csv, n_npz) == (96, 48), \
    "campaign incomplete - rerun Cell 3 to resume, then rerun this cell"
with open("results/MANIFEST_sha256.txt", "w") as m:
    for f in files:
        m.write(hashlib.sha256(open(f, 'rb').read()).hexdigest()[:16]
                + "  " + os.path.basename(f) + "\n")
zn = f"discrepancy_campaign_R4_FULL_results_{datetime.date.today()}.zip"
with zipfile.ZipFile(zn, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files + ["results/MANIFEST_sha256.txt"]:
        z.write(f, arcname=os.path.basename(f))
print("wrote", zn, "-", len(files), "files + manifest")
try:
    from google.colab import files as gf
    gf.download(zn)
except ImportError:
    print("(not on Colab - zip is in the working directory)")


inventory: 96 CSVs + 48 snapshots = 144
wrote discrepancy_campaign_R4_FULL_results_2026-08-12.zip - 144 files + manifest


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Return path

Upload the results zip back to the chat. Analysis proceeds strictly
against the registered quantities: window-means over W* = [20, 40] per
rung, ladder slopes with 95% log-space t CIs across replicates,
falsifiers A/B for M4a/M4b, the secondary M-vs-C4 contrast against the
frozen r3 curves of record, and the S-arm D_v snapshot criterion.
MANIFEST must verify 144/144. No campaign offset is ever reused.
